# Deliverable 3 — Dataset Creation Pipeline

**Purpose**: Build unified train/test feature checkpoints for all four models (Lasso OLS, FFNN, Random Forest, XGBoost).

**Design**: one unified parquet per dataset size holds every column any model family needs — numerics, string categoricals, StringIndexer-indexed categoricals, and OneHotEncoded categoricals side by side. Each downstream model notebook selects the columns appropriate for its family (trees use `_idx`, linear uses `_oh`); Parquet's column pruning means unused columns are never read.

**Outputs** (per `DATASET_SIZE` in `{"12m", "60m"}`):
- `features_raw_{size}.parquet` — post-FE, pre-encoding intermediate (skip-if-exists)
- `features_{size}_train.parquet` — train side, fully encoded
- `features_{size}_test.parquet` — test side, fully encoded (encoders fit on train only)

**Feature-family split**:
- Both families share cyclical time (`dep_sin`, `dep_cos`, `month_sin`, `month_cos`) and all numerics/flags.
- Only difference: categoricals are `_idx` for trees, `_oh` for linear. `DAY_OF_WEEK` stays raw for trees, OHE for linear.

## 0. Imports

In [0]:
import math
import itertools
from datetime import timedelta, datetime
from math import pi

import numpy as np
import pandas as pd
import networkx as nx
import holidays as pyholidays
import matplotlib.pyplot as plt

from pyspark.sql import functions as F, types as T, Row
from pyspark.sql.window import Window
from pyspark.ml import Pipeline, Transformer
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, Imputer,
    Bucketizer, StandardScaler, RobustScaler, MinMaxScaler,
)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

## 1. Configuration

In [0]:
data_BASE_DIR = "dbfs:/mnt/mids-w261/"
display(dbutils.fs.ls(f"{data_BASE_DIR}"))

path,name,size,modificationTime
dbfs:/mnt/mids-w261/Data/,Data/,0,1785440083226
dbfs:/mnt/mids-w261/HW5/,HW5/,0,1785440083226
dbfs:/mnt/mids-w261/OTPW_12M/,OTPW_12M/,0,1785440083226
dbfs:/mnt/mids-w261/OTPW_1D_CSV/,OTPW_1D_CSV/,0,1785440083226
dbfs:/mnt/mids-w261/OTPW_36M/,OTPW_36M/,0,1785440083227
dbfs:/mnt/mids-w261/OTPW_3M/,OTPW_3M/,0,1785440083227
dbfs:/mnt/mids-w261/OTPW_3M_2015.csv,OTPW_3M_2015.csv,1500620247,1741625185000
dbfs:/mnt/mids-w261/OTPW_3M_2015_delta/,OTPW_3M_2015_delta/,0,1785440083227
dbfs:/mnt/mids-w261/OTPW_60M/,OTPW_60M/,0,1785440083227
dbfs:/mnt/mids-w261/OTPW_60M_Backup/,OTPW_60M_Backup/,0,1785440083227


### Quick verification of data: comparing 2015 data from 60M_backup to 1 yr 2015 data to ensure records match

In [0]:
df_backup = spark.read.parquet("dbfs:/mnt/mids-w261/OTPW_60M_Backup/")
df_2015_backup = df_backup.filter(F.year(F.to_date("FL_DATE")) == 2015)
df_2015_gold   = spark.read.parquet("dbfs:/mnt/mids-w261/OTPW_12M/OTPW_12M_2015_parquet/")

print("Row count (gold):", df_2015_gold.count())
print("Row count (backup):", df_2015_backup.count())
print("Row count difference (backup-gold):", df_2015_backup.count() - df_2015_gold.count())

# Schema diff
print("\nColumns in gold but not backup:", set(df_2015_gold.columns) - set(df_2015_backup.columns))
print("Columns in backup but not gold:", set(df_2015_backup.columns) - set(df_2015_gold.columns))

print("\n See below cell below for deep dive - in backup, have column sched_depart_date_time_UTC which is what we use for feature engineering in this pipeline. sched_depart_date_time, not in backup dataset, is in airport local time, which we do not use for any feature engineering steps (we only use sched_depart_date_time_UTC).")

# Key stat parity — these should be nearly identical if backup 2015 is clean
for c in ["DEP_DELAY", "ARR_DELAY", "DISTANCE"]:
    g = df_2015_gold.select(F.mean(c), F.stddev(c), F.count(F.when(F.col(c).isNull(), 1))).collect()
    b = df_2015_backup.select(F.mean(c), F.stddev(c), F.count(F.when(F.col(c).isNull(), 1))).collect()
    print(c, "gold:", g, "backup:", b)

# Row-level match on a natural key, not just aggregates
key_cols = ["FL_DATE", "OP_CARRIER_FL_NUM", "ORIGIN", "DEST", "CRS_DEP_TIME"]
g_keys = df_2015_gold.select(*key_cols).distinct()
b_keys = df_2015_backup.select(*key_cols).distinct()
print("\nRows in gold but not backup:", g_keys.subtract(b_keys).count())
print("Rows in backup but not gold:", b_keys.subtract(g_keys).count())

print("\nDuplicates in backup 12m 2025 sample:", df_2015_backup.groupBy(*key_cols).count().filter(F.col("count") > 1).count())
print("Duplicates in gold 12m 2025 sample:", df_2015_gold.groupBy(*key_cols).count().filter(F.col("count") > 1).count())

print("\nSee cells below for deep dive - 388 rows in Backup not in 12M that ARRIVED in 2016 so not counted in 2015 12M. Besides this, datasets appear to match each other.")

Row count (gold): 5811854
Row count (backup): 5812242
Row count difference (backup-gold): 388

Columns in gold but not backup: {'_row_desc', 'sched_depart_date_time'}
Columns in backup but not gold: set()

 See below cell below for deep dive - in backup, have column sched_depart_date_time_UTC which is what we use for feature engineering in this pipeline. sched_depart_date_time, not in backup dataset, is in airport local time, which we do not use for any feature engineering steps (we only use sched_depart_date_time_UTC).
DEP_DELAY gold: [Row(avg(DEP_DELAY)=9.365873559916134, stddev(DEP_DELAY)=37.065179278827564, count(CASE WHEN (DEP_DELAY IS NULL) THEN 1 END)=86059)] backup: [Row(avg(DEP_DELAY)=9.365468887553503, stddev(DEP_DELAY)=37.06516868618217, count(CASE WHEN (DEP_DELAY IS NULL) THEN 1 END)=86061)]
ARR_DELAY gold: [Row(avg(ARR_DELAY)=4.405822402682513, stddev(ARR_DELAY)=39.25774353490936, count(CASE WHEN (ARR_DELAY IS NULL) THEN 1 END)=104966)] backup: [Row(avg(ARR_DELAY)=4.404920

In [0]:
display(spark.createDataFrame([(col,) for col in df_2015_backup.columns], ['Column']))

Column
QUARTER
DAY_OF_MONTH
DAY_OF_WEEK
FL_DATE
OP_UNIQUE_CARRIER
OP_CARRIER_AIRLINE_ID
OP_CARRIER
TAIL_NUM
OP_CARRIER_FL_NUM
ORIGIN_AIRPORT_ID


In [0]:
display(df_2015_backup[["sched_depart_date_time_UTC"]].join(df_2015_gold[["sched_depart_date_time"]], how='inner'))

sched_depart_date_time_UTC,sched_depart_date_time
2015-07-10T13:55:00,2015-06-17T12:14:00Z
2015-07-10T13:55:00,2015-11-01T10:31:00Z
2015-07-10T13:55:00,2015-02-12T10:29:00Z
2015-07-10T13:55:00,2015-02-10T18:05:00Z
2015-07-10T13:55:00,2015-02-13T16:10:00Z
2015-07-10T13:55:00,2015-04-12T18:03:00Z
2015-07-10T13:55:00,2015-06-28T15:50:00Z
2015-07-10T13:55:00,2015-08-25T15:58:00Z
2015-07-10T13:55:00,2015-02-12T09:40:00Z
2015-07-10T13:55:00,2015-03-06T09:43:00Z


In [0]:
display(b_keys.subtract(g_keys))

FL_DATE,OP_CARRIER_FL_NUM,ORIGIN,DEST,CRS_DEP_TIME
2015-12-31,448,LGB,OAK,2015
2015-12-31,223,HNL,LIH,1943
2015-12-31,867,LAX,EWR,2354
2015-12-31,5213,DEN,RNO,2210
2015-12-31,1521,HNL,SLC,2100
2015-12-31,1933,HNL,SEA,2300
2015-12-31,1281,OGG,LAX,2140
2015-12-31,808,DEN,PBI,2355
2015-12-31,384,PDX,SFO,2135
2015-12-31,545,OGG,HNL,1825


In [0]:
# ================================ CHANGE ME ================================
DATASET_SIZE = "12m"       # "12m" for 1-year (2015), "60m" for 5-year (2015-2019)
SECTION      = "1"
NUMBER       = "1"
REBUILD_RAW  = True       # True = rerun expensive FE even if intermediate exists
# ============================================================================

# --- Dataset-specific config -----------------------------------------------
CONFIGS = {
    "12m": {
        "otpw_path":      "dbfs:/mnt/mids-w261/OTPW_12M/OTPW_12M_2015_parquet/",
        "date_start":     "2015-01-01",
        "date_end":       "2015-12-31",
        "holiday_years":  range(2014, 2017),
    },
    "60m": {
        "otpw_path":      "dbfs:/mnt/mids-w261/OTPW_60M/OTPW_60M_Backup/",
        "date_start":     "2015-01-01",
        "date_end":       "2019-12-31",
        "holiday_years":  range(2014, 2021),
    },
}
CFG = CONFIGS[DATASET_SIZE]

# The '_1y' weather file is 2019 data (wrong year for 12M); use the multi-year
# archive filtered to CFG['date_start']..CFG['date_end'].
WEATHER_ARCHIVE = "dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_weather_data/"

# --- Output paths ----------------------------------------------------------
FOLDER      = f"dbfs:/student-groups/Group_{SECTION}_{NUMBER}/rolling"
RAW_PATH    = f"{FOLDER}/features_raw_{DATASET_SIZE}.parquet"
TRAIN_PATH  = f"{FOLDER}/features_{DATASET_SIZE}_train.parquet"
TEST_PATH   = f"{FOLDER}/features_{DATASET_SIZE}_test.parquet"

spark.sparkContext.setCheckpointDir(f"{FOLDER}/_checkpoints")

print(f"Dataset size:   {DATASET_SIZE}")
print(f"OTPW path:      {CFG['otpw_path']}")
print(f"Date range:     {CFG['date_start']} to {CFG['date_end']}")
print(f"Raw output:     {RAW_PATH}")
print(f"Train output:   {TRAIN_PATH}")
print(f"Test output:    {TEST_PATH}")

# --- Modeling constants ----------------------------------------------------
label_col        = "DEP_DELAY"
CEILING_SENTINEL = 99999.0
VALID_COVERAGE   = {"CLR", "FEW", "SCT", "BKN", "OVC", "VV", "10"}
INHG_TO_MB       = 33.8639
ROLL_LOOKBACK_HOURS = 26
ROLL_END_HOURS      = 2
ROLL_BASE_COLS = [
    "HourlyVisibility", "HourlyWindSpeed", "HourlyDryBulbTemperature",
    "HourlyDewPointTemperature", "HourlyAltimeterSetting",
]

_WEATHER_COLS = [
    "HourlyPrecipitation", "HourlyVisibility", "HourlyWindSpeed",
    "HourlyDryBulbTemperature", "HourlyDewPointTemperature",
    "HourlySkyConditions", "HourlyAltimeterSetting",
    "HourlyPresentWeatherType",
]

layer_schema = T.ArrayType(T.StructType([
    T.StructField("coverage",  T.StringType()),
    T.StructField("oktas",     T.IntegerType()),
    T.StructField("height_ft", T.IntegerType()),
]))

Dataset size:   12m
OTPW path:      dbfs:/mnt/mids-w261/OTPW_12M/OTPW_12M_2015_parquet/
Date range:     2015-01-01 to 2015-12-31
Raw output:     dbfs:/student-groups/Group_1_1/rolling/features_raw_12m.parquet
Train output:   dbfs:/student-groups/Group_1_1/rolling/features_12m_train.parquet
Test output:    dbfs:/student-groups/Group_1_1/rolling/features_12m_test.parquet


## 2. Feature groups

Column-group definitions. These drive both the feature-engineering orchestrator and the encoder-fitting step below. 

In [0]:
# --- Numeric features shared across all four models ------------------------
schedule_cols  = ["CRS_ARR_TIME", "CRS_ELAPSED_TIME"]
flight_cols    = ["leg_num", "DISTANCE", "prev_leg_dep_delay"]

# ALL models use cyclical encodings for departure time and month; there is no separate tree_time_numeric feature list anymore.
time_hol_numeric = [
    "dep_sin", "dep_cos", "is_weekend", "month_sin", "month_cos",
    "holiday_intensity", "days_to_holiday", "is_before_holiday",
]

graphy_cols = [
    "ORIGIN_inDegree", "ORIGIN_outDegree", "ORIGIN_degree",
    "ORIGIN_pageRank", "ORIGIN_betweenness", "ORIGIN_closeness",
    "DEST_inDegree",   "DEST_outDegree",   "DEST_degree",
    "DEST_pageRank",   "DEST_betweenness", "DEST_closeness",
]

# Cleaned numeric weather columns (post WeatherCleaner).
weather_num_cols = [
    "HourlyPrecipitation_2h", "HourlyPrecipitation_8h", "dest_HourlyPrecipitation_2h",
    "HourlyVisibility_2h",    "HourlyVisibility_8h",    "dest_HourlyVisibility_2h",
    "HourlyWindSpeed_2h",     "HourlyWindSpeed_8h",     "dest_HourlyWindSpeed_2h",
    "HourlyDryBulbTemperature_2h", "HourlyDryBulbTemperature_8h", "dest_HourlyDryBulbTemperature_2h",
    "HourlyDewPointTemperature_2h", "HourlyDewPointTemperature_8h", "dest_HourlyDewPointTemperature_2h",
    "HourlyAltimeterSetting_2h",   "HourlyAltimeterSetting_8h",   "dest_HourlyAltimeterSetting_2h",
]

# Numeric weather subject to median imputation (linear models). Trees use raw + missing-flags.
weather_impute_cols = [
    "HourlyVisibility_2h", "HourlyWindSpeed_2h", "spread_2h",
    "dest_HourlyVisibility_2h", "dest_HourlyWindSpeed_2h",
]

# 0/1 flags produced by FE transformers (missing/obscured/frozen/etc.).
weather_flag_cols = [
    "sky_missing",  "dest_sky_missing",
    "sky_obscured", "dest_sky_obscured",
    "sky_went_ifr",
    "precip_missing", "dest_precip_missing",
    "precip_thunderstorm", "dest_precip_thunderstorm",
    "precip_frozen",      "dest_precip_frozen",
]

# Continuous weather trend features.
weather_trend_numeric = [
    "precip_delta_8h_to_2h", "sky_ceiling_delta_ft", "sky_vis_delta",
    "alt_delta_mb", "precip_accum", "precip_hours_wet",
]

# Continuous weather columns used by linear models AS-IS (no bucketing).
weather_lin_continuous = [
    "sky_ceiling_ft", "dest_sky_ceiling_ft", "spread_2h", "alt_delta_mb",
]

# Weather categoricals (strings) that need indexing/OHE.
weather_cat_cols = [
    "sky_last_coverage", "dest_sky_last_coverage",
    "sky_ceiling_transition", "precip_bucket", "dest_precip_bucket",
]

# Non-weather categoricals.
other_cat_cols = ["ORIGIN", "dep_category"]

# Buckets applied ONLY for linear models (trees do this natively via splits).
# Each entry is (input_col, output_bin_col, splits).
BUCKET_SPECS = [
    ("spread_2h",         "spread_bin",          [-float("inf"), 3, 8, 13, float("inf")]),
    ("sky_ceiling_ft",        "sky_ceiling_bin",     [0, 500, 1000, 3000, float("inf")]),
    ("dest_sky_ceiling_ft",   "dest_sky_ceiling_bin",[0, 500, 1000, 3000, float("inf")]),
    ("precip_delta_8h_to_2h", "precip_delta_bin",    [-float("inf"), -0.1, 0.1, float("inf")]),
]
weather_bucket_bins = [spec[1] for spec in BUCKET_SPECS]
weather_bucket_ohs  = [b.replace("_bin", "_oh") for b in weather_bucket_bins]

# Every string categorical column that trees need indexed. `DAY_OF_WEEK` stays
# raw because it's already 1-7 (Mon-Sun); no need to index a numeric column.
STRING_CAT_COLS = weather_cat_cols + other_cat_cols

# Identity/key columns kept in the checkpoint for post-hoc analysis + CV splits.
identity_cols = [
    "sched_depart_date_time_UTC", "FL_DATE",
    "OP_UNIQUE_CARRIER", "OP_CARRIER_FL_NUM",
    "ORIGIN", "DEST", label_col,
]

## 3. Load raw data + build holiday calendar

In [0]:
# =============================================================================
# Load OTPW parquet files from PartitionId=0-4, parts 00000 through 00009
# =============================================================================
OTPW_BASE = "dbfs:/mnt/mids-w261/OTPW_60M_Backup"

# Read all matched parquet files and order by FL_DATE
df_otpw_raw = (
    spark.read.parquet(f"{OTPW_BASE}/*")
         .orderBy("FL_DATE")
         .filter((F.col("FL_DATE") >= CFG["date_start"]) & (F.col("FL_DATE") <= CFG["date_end"]))
)

# Weather archive, filtered to the same date range
df_weather = (
   spark.read.parquet(WEATHER_ARCHIVE)
         .filter((F.col("DATE") >= CFG["date_start"]) & (F.col("DATE") <= CFG["date_end"]))
)

print(f"Raw OTPW rows:    {df_otpw_raw.count():,}")
print(f"Weather rows:     {df_weather.count():,}")

Raw OTPW rows:    5,812,242
Weather rows:     123,511,870


In [0]:
# ============================================================================
# Holiday calendar (deterministic, date-level)
# ============================================================================

def build_holiday_calendar(years):
    """One row per date with holiday_intensity (0-3), days_to_holiday (0-7),
    and is_before_holiday (0/1). Extend the year range beyond the data window
    so New Year edge cases are covered.
    """
    years = list(years)
    cal   = pd.DataFrame({"d": pd.date_range(f"{years[0]}-01-01", f"{years[-1]}-12-31", freq="D")})
    us    = pyholidays.US(years=years)

    TRAVEL_WINDOWS = {
        "thanksgiving": (3, 5, 4),
        "memorial":     (2, 3, 2),
        "labor":        (2, 3, 3),
        "independence": (2, 3, 4),
        "washington":   (1, 3, 1),
    }

    intensity = {}
    holiday_dates = []

    def bump(day, val):
        intensity[day] = max(intensity.get(day, 0), val)

    for day, name in us.items():
        day = pd.Timestamp(day)
        n = name.lower()
        cfg = next((c for key, c in TRAVEL_WINDOWS.items() if key in n), None)
        if cfg:
            tier, before, after = cfg
            for o in range(-before, after + 1):
                bump(day + timedelta(days=o), tier)
            holiday_dates.append(day)
        elif "christmas" in n or "new year" in n:
            holiday_dates.append(day)

    # Christmas/New Year: one continuous tier-3 block 12/20 -> 1/6
    for y in years:
        d, end = pd.Timestamp(f"{y}-12-20"), pd.Timestamp(f"{y+1}-01-06")
        while d <= end:
            bump(d, 3)
            d += timedelta(days=1)

    cal["FL_DATE"]           = cal.d.dt.strftime("%Y-%m-%d")
    cal["holiday_intensity"] = cal.d.map(lambda x: intensity.get(x, 0))

    hol      = pd.to_datetime(sorted(holiday_dates)).values.astype("datetime64[D]")
    cal_days = cal["d"].values.astype("datetime64[D]")
    diff     = (cal_days[:, None] - hol[None, :]) / np.timedelta64(1, "D")
    nearest  = diff[np.arange(len(diff)), np.argmin(np.abs(diff), axis=1)]

    cal["days_to_holiday"]   = np.minimum(np.abs(nearest), 7)
    cal["is_before_holiday"] = ((nearest < 0) & (np.abs(nearest) <= 7)).astype(int)

    return spark.createDataFrame(
        cal[["FL_DATE", "holiday_intensity", "days_to_holiday", "is_before_holiday"]]
    )


cal_df = build_holiday_calendar(CFG["holiday_years"])
display(cal_df.limit(5))

FL_DATE,holiday_intensity,days_to_holiday,is_before_holiday
2014-01-01,0,0.0,0
2014-01-02,0,1.0,0
2014-01-03,0,2.0,0
2014-01-04,0,3.0,0
2014-01-05,0,4.0,0


## 4. Feature-engineering helpers


In [0]:
# ============================================================================
# Type normalization
# ============================================================================
def normalize_input_types(df):
    """CSV/parquet compatibility: FL_DATE -> 'yyyy-MM-dd' string; dest_station_id -> string."""
    df = df.withColumn("FL_DATE", F.date_format(F.to_date(F.col("FL_DATE")), "yyyy-MM-dd"))
    if "dest_station_id" in df.columns:
        df = df.withColumn("dest_station_id", F.col("dest_station_id").cast("string"))
    return df


def remove_null_targets(df):
    """Drop rows where DEP_DELAY is null (cancelled/diverted flights). BUG FIX:
    the original version discarded the .filter() result.
    """
    return df.filter(F.col("DEP_DELAY").isNotNull())


# ============================================================================
# Time / calendar features
# ============================================================================
def add_departure_time_features(df):
    t = F.col("CRS_DEP_TIME").cast("int")
    m = F.month(F.to_date(F.col("FL_DATE")))
    return (df
        .withColumn("dep_hour", (t / 100).cast("int") % 24)
        .withColumn("dep_min",  (t % 100).cast("int"))
        .withColumn("dep_minofday", ((t / 100).cast("int") % 24) * 60 + (t % 100).cast("int"))
        .withColumn("dep_sin", F.sin(2 * F.lit(pi) * F.col("dep_minofday") / 1440))
        .withColumn("dep_cos", F.cos(2 * F.lit(pi) * F.col("dep_minofday") / 1440))
        .withColumn("dep_category",
            F.when(F.col("dep_hour") < 5,  "red_eye")
             .when(F.col("dep_hour") < 12, "morning")
             .when(F.col("dep_hour") < 17, "afternoon")
             .when(F.col("dep_hour") < 21, "evening")
             .otherwise("night"))
        .withColumn("is_weekend", (F.col("DAY_OF_WEEK").cast("int") >= 6).cast("int"))
        .withColumn("month", m)
        .withColumn("month_sin", F.sin(2 * F.lit(pi) * m / 12))
        .withColumn("month_cos", F.cos(2 * F.lit(pi) * m / 12)))


def add_holiday_features(df, cal_df_broadcast):
    return (df
        .drop("holiday_intensity", "days_to_holiday", "is_before_holiday")
        .join(F.broadcast(cal_df_broadcast), on="FL_DATE", how="left")
        .fillna({"holiday_intensity": 0, "days_to_holiday": 7, "is_before_holiday": 0}))


# ============================================================================
# Weather attachment (origin + destination, 2h and 8h snapshots)
# ============================================================================
def add_weather_cols(df, df_weather_src):
    """Attach origin/destination weather from [8h, 2h] pre-departure window.
    Uses bucketed equi-join (station, truncated_hour) + exact-window guard.
    """
    df_flights_prep = (
        df
        .withColumn("_flight_id", F.sha2(F.concat_ws("|",
            F.col("OP_UNIQUE_CARRIER"), F.col("OP_CARRIER_FL_NUM"),
            F.col("ORIGIN"), F.col("DEST"),
            F.col("sched_depart_date_time_UTC").cast("string")), 256))
        .withColumn("dest_station_id", F.col("dest_station_id").cast("string"))
        .withColumn("STATION",         F.col("STATION").cast("string"))
        .withColumnRenamed("two_hours_prior_depart_UTC", "_t_two_prior")
        .withColumn("_t_eight_prior", F.expr("sched_depart_date_time_UTC - INTERVAL 8 HOURS"))
    ).cache()
    df_flights_prep.count()

    origin_stations = (df_flights_prep.select(F.col("STATION"))
                       .filter(F.col("STATION").isNotNull()).distinct())
    dest_stations = (df_flights_prep.select(F.col("dest_station_id").alias("STATION"))
                     .filter(F.col("STATION").isNotNull()).distinct())

    def _filter_weather(stations):
        return (df_weather_src
            .filter(F.col("REPORT_TYPE").isin("FM-15", "FM-16"))
            .join(stations.hint("broadcast"), on="STATION", how="inner")
            .withColumn("DATE", F.to_timestamp("DATE"))
            .withColumn("_weather_hour", F.date_trunc("hour", F.col("DATE"))))

    df_weather_filtered_dest   = _filter_weather(dest_stations)
    df_weather_filtered_origin = _filter_weather(origin_stations)

    df_flights_bucketed = (df_flights_prep
        .withColumn("_buckets", F.expr(
            "sequence(date_trunc('hour', _t_eight_prior),"
            " date_trunc('hour', _t_two_prior), interval 1 hour)"))
        .withColumn("_bucket_hour", F.explode("_buckets"))
        .drop("_buckets"))

    def _match(weather_df, station_key, cols):
        return (df_flights_bucketed.alias("f")
            .join(weather_df.alias("w"),
                on=((F.col(f"f.{station_key}") == F.col("w.STATION")) &
                    (F.col("f._bucket_hour") == F.col("w._weather_hour"))),
                how="inner")
            .filter(F.col("w.DATE").between(F.col("f._t_eight_prior"),
                                            F.col("f._t_two_prior")))
            .select(F.col("f._flight_id").alias("_flight_id"),
                    F.col("w.DATE").alias("_obs_date"),
                    *[F.col(f"w.{c}").alias(c) for c in cols]))

    df_matched_dest   = _match(df_weather_filtered_dest,   "dest_station_id", _WEATHER_COLS)
    df_matched_origin = _match(df_weather_filtered_origin, "STATION",         _WEATHER_COLS)

    w_near_t2 = Window.partitionBy("_flight_id").orderBy(F.col("_obs_date").desc())
    w_near_t8 = Window.partitionBy("_flight_id").orderBy(F.col("_obs_date").asc())

    def _snapshots(matched, cols, prefix):
        ranked = (matched
            .withColumn("_rn_t2", F.row_number().over(w_near_t2))
            .withColumn("_rn_t8", F.row_number().over(w_near_t8)))
        s2 = (ranked.filter("_rn_t2 = 1")
            .select("_flight_id",
                    F.col("_obs_date").alias(f"{prefix}obs_ts_2h"),
                    *[F.col(c).alias(f"{prefix}{c}_2h") for c in cols]))
        s8 = (ranked.filter("_rn_t8 = 1")
            .select("_flight_id",
                    F.col("_obs_date").alias(f"{prefix}obs_ts_8h"),
                    *[F.col(c).alias(f"{prefix}{c}_8h") for c in cols]))
        return s2, s8

    df_2h_dest,   df_8h_dest   = _snapshots(df_matched_dest,   _WEATHER_COLS, "dest_")
    df_2h_origin, df_8h_origin = _snapshots(df_matched_origin, _WEATHER_COLS, "")

    return (df_flights_prep
        .join(df_2h_dest,   "_flight_id", "left").join(df_8h_dest,   "_flight_id", "left")
        .join(df_2h_origin, "_flight_id", "left").join(df_8h_origin, "_flight_id", "left"))


# ============================================================================
# Aircraft-leg features (previous leg's departure delay, position in day)
# ============================================================================
def add_leg_features(df):
    leg_window = Window.partitionBy("TAIL_NUM", "FL_DATE").orderBy(F.col("CRS_DEP_TIME").cast("int"))
    prev_delay = F.lag("DEP_DELAY", 1).over(leg_window)

    df = (df
        .withColumn("leg_num", F.row_number().over(leg_window))
        .withColumn("prev_leg_actual_dep_ts",
                    F.lag("sched_depart_date_time_UTC", 1).over(leg_window)
                    + (prev_delay * F.expr("INTERVAL 1 MINUTE")))
        .withColumn("prediction_cutoff_ts",
                    F.col("sched_depart_date_time_UTC") - F.expr("INTERVAL 2 HOURS"))
        .withColumn("prev_leg_dep_delay",
                    F.when(F.col("prev_leg_actual_dep_ts") <= F.col("prediction_cutoff_ts"), prev_delay)
                     .otherwise(F.lit(None).cast("double"))))
    helper_cols = [c for c in df.columns if c not in ["prev_leg_actual_dep_ts", "prediction_cutoff_ts"]]
    return df.select(*helper_cols)


# ============================================================================
# Weather numeric cleaning (T=trace, *=missing, trailing 's' suffix)
# ============================================================================
def clean_weather_num(weather_num_cols, df):
    trace_flag_substr = "Precipitation"
    for c in weather_num_cols:
        if c not in df.columns:
            print(f"WeatherCleaner: skipping absent column {c}")
            continue
        col = F.trim(F.col(c))
        is_trace = col == "T"
        if trace_flag_substr and trace_flag_substr in c:
            df = df.withColumn(f"{c}_was_trace", F.coalesce(is_trace, F.lit(False)).cast("int"))
        df = df.withColumn(c,
            F.when(col.isNull(),        F.lit(None).cast("double"))
             .when(col.contains("*"),   F.lit(None).cast("double"))
             .when(col == "",           F.lit(None).cast("double"))
             .when(col == "T",          F.lit(0.0))
             .when(col.rlike(r"^-?\d*\.?\d+s?$"),
                   F.regexp_replace(col, "s$", "").cast("double"))
             .otherwise(F.lit(None).cast("double")))
    return df


# ============================================================================
# Precipitation features
# ============================================================================
def process_precip_features(df, precip_cols, wx_col, temp_col, prefix):
    newest = precip_cols[-1]
    df = df.withColumn(f"{prefix}missing", F.col(newest).isNull())
    ps = [F.coalesce(F.col(c), F.lit(0.0)) for c in precip_cols]
    p_new = ps[-1]
    wx = F.coalesce(F.col(wx_col), F.lit(""))
    trace_new = F.coalesce(F.col(f"{newest}_was_trace"), F.lit(0)) == 1

    df = df.withColumn(f"{prefix}bucket",
        F.when(trace_new, "TRACE")
         .when(p_new == 0,       "NONE")
         .when(p_new <= 0.10,    "LIGHT")
         .when(p_new <= 0.30,    "MODERATE")
         .otherwise("HEAVY"))

    if len(precip_cols) >= 2:
        accum = ps[0]
        for x in ps[1:]:
            accum = accum + x
        df = (df.withColumn(f"{prefix}accum", accum)
                .withColumn(f"{prefix}hours_wet", sum((x > 0).cast("int") for x in ps)))
        deltas = []
        for older, newer in zip(precip_cols, precip_cols[1:]):
            name = f"{prefix}delta_{older.split('_')[-1]}_to_{newer.split('_')[-1]}"
            d = (F.when(F.col(newer).isNull() | F.col(older).isNull(), F.lit(None).cast("double"))
                  .otherwise(F.coalesce(F.col(newer), F.lit(0.0)) - F.coalesce(F.col(older), F.lit(0.0))))
            df = df.withColumn(name, d)
            deltas.append(name)
        if len(deltas) >= 2:
            df = df.withColumn(f"{prefix}accel", F.col(deltas[-1]) - F.col(deltas[-2]))

    return (df
        .withColumn(f"{prefix}thunderstorm", wx.contains("TS").cast("int"))
        .withColumn(f"{prefix}frozen",
            (wx.rlike("SN|FZRA|FZDZ|PL|IC|GS|GR")
             | ((p_new > 0) & (wx == "") & (F.col(temp_col) <= 34))
            ).cast("int")))


def get_precip_features(df):
    df = process_precip_features(df,
        precip_cols=["HourlyPrecipitation_8h", "HourlyPrecipitation_2h"],
        wx_col="HourlyPresentWeatherType_2h",
        temp_col="HourlyDryBulbTemperature_2h", prefix="precip_")
    df = process_precip_features(df,
        precip_cols=["dest_HourlyPrecipitation_2h"],
        wx_col="dest_HourlyPresentWeatherType_2h",
        temp_col="dest_HourlyDryBulbTemperature_2h", prefix="dest_precip_")
    return df


# ============================================================================
# Sky features
# ============================================================================
@F.udf(layer_schema)
def parse_sky(s):
    if s is None:
        return None
    s = s.strip()
    if s in ("", "*", "M"):
        return None
    tokens, layers, i = s.split(), [], 0
    while i < len(tokens):
        t = tokens[i]
        if ":" in t:
            cov, _, oktas = t.partition(":")
            cov = cov if cov in VALID_COVERAGE else None
            oktas_val = int(oktas) if oktas.isdigit() else None
            height = None
            if i + 1 < len(tokens) and tokens[i + 1].isdigit():
                height = int(tokens[i + 1]) * 100
                i += 1
            layers.append((cov, oktas_val, height))
        elif t.isdigit():
            layers.append((None, None, int(t) * 100))
        i += 1
    return layers if layers else None


def _ceiling_expr(L):
    return F.expr(
        f"""array_min(transform(
              filter({L}, x -> x.coverage IN ('BKN','OVC','VV')
                               AND x.height_ft IS NOT NULL),
              x -> x.height_ft))""")


def process_sky_features(df, raw_col, prefix):
    L = f"{prefix}layers"
    df = df.withColumn(L, parse_sky(F.col(raw_col)))
    last_coverage = F.expr(
        f"try_element_at(filter({L}, x -> x.coverage IS NOT NULL "
        f"AND x.coverage <> '10').coverage, -1)")
    return (df
        .withColumn(f"{prefix}missing", F.col(L).isNull())
        .withColumn(f"{prefix}ceiling_ft", _ceiling_expr(L))
        .withColumn(f"{prefix}last_coverage", last_coverage)
        .withColumn(f"{prefix}obscured",
                    F.coalesce(F.expr(f"exists({L}, x -> x.coverage = 'VV' OR x.oktas = 9)"),
                               F.lit(False)).cast("int"))
        .drop(L))


def add_ceiling_only(df, raw_col, out_col):
    L = "_tmp_layers"
    df = df.withColumn(L, parse_sky(F.col(raw_col)))
    df = df.withColumn(out_col, _ceiling_expr(L))
    df = df.withColumn(f"{out_col}_src_missing", F.col(L).isNull())
    return df.drop(L)


def add_sky_trends(df, prefix, ceil_new, ceil_old, old_missing, vis_new, vis_old):
    cn, co = F.col(ceil_new), F.col(ceil_old)
    df = (df
        .withColumn(f"{prefix}ceiling_delta_ft",
                    F.when(cn.isNotNull() & co.isNotNull(), cn - co)
                     .otherwise(F.lit(None).cast("double")))
        .withColumn(f"{prefix}ceiling_transition",
                    F.when(F.col(old_missing) | F.col(f"{prefix}missing"), "UNKNOWN")
                     .when(co.isNull()    & cn.isNull(),    "NONE_NONE")
                     .when(co.isNull()    & cn.isNotNull(), "FORMED")
                     .when(co.isNotNull() & cn.isNull(),    "LIFTED")
                     .when(cn - co <= -500, "DROPPING")
                     .when(cn - co >=  500, "RISING")
                     .otherwise("STEADY"))
        .withColumn(f"{prefix}went_ifr",
                    ((co.isNull() | (co >= 1000)) & cn.isNotNull() & (cn < 1000)).cast("int")))
    vn, vo = F.col(vis_new), F.col(vis_old)
    df = df.withColumn(f"{prefix}vis_delta",
                       F.when(vn.isNotNull() & vo.isNotNull(), vn - vo)
                        .otherwise(F.lit(None).cast("double")))
    return df


def get_sky_features(df):
    df = process_sky_features(df, "HourlySkyConditions_2h", "sky_")
    df = add_ceiling_only(df, "HourlySkyConditions_8h", "sky_ceiling_ft_8h")
    df = add_sky_trends(df, "sky_",
        ceil_new="sky_ceiling_ft", ceil_old="sky_ceiling_ft_8h",
        old_missing="sky_ceiling_ft_8h_src_missing",
        vis_new="HourlyVisibility_2h", vis_old="HourlyVisibility_8h")
    df = process_sky_features(df, "dest_HourlySkyConditions_2h", "dest_sky_")
    return df


# ============================================================================
# Pressure + spread
# ============================================================================
def get_pressure_spread(df):
    return (df
        .withColumn("spread_2h",
                    F.col("HourlyDryBulbTemperature_2h") - F.col("HourlyDewPointTemperature_2h"))
        .withColumn("alt_delta_mb",
                    (F.col("HourlyAltimeterSetting_2h") - F.col("HourlyAltimeterSetting_8h")) * INHG_TO_MB)
        .withColumn("alt_delta_mb",
                    F.when(F.abs(F.col("alt_delta_mb")) <= 15, F.col("alt_delta_mb"))
                     .otherwise(F.lit(None).cast("double"))))


# ============================================================================
# Rolling weather medians (trailing [26h, 2h] per-station)
# ============================================================================
def add_rolling_weather_medians(df, df_weather_src, station_col, prefix):
    df = (df
        .withColumn("_t_end",   F.col("sched_depart_date_time_UTC") - F.expr("INTERVAL 2 HOURS"))
        .withColumn("_t_start", F.col("_t_end") - F.expr(f"interval {ROLL_LOOKBACK_HOURS - ROLL_END_HOURS} hours")))
    df = df.persist()
    df.count()

    stations = (df.select(F.col(station_col).cast("string").alias("STATION"))
                  .filter(F.col("STATION").isNotNull()).distinct())

    wsrc = (df_weather_src
        .filter(F.col("REPORT_TYPE").isin("FM-15", "FM-16"))
        .join(F.broadcast(stations), on="STATION", how="inner"))
    wsrc = clean_weather_num(ROLL_BASE_COLS, wsrc)
    wsrc = (wsrc
        .withColumn("_obs_date", F.to_timestamp("DATE"))
        .withColumn("_weather_hour", F.date_trunc("hour", F.col("_obs_date")))
        .select("STATION", "_obs_date", "_weather_hour", *ROLL_BASE_COLS))

    bucketed = (df
        .filter(F.col("_t_start").isNotNull() & F.col("_t_end").isNotNull()
                & F.col(station_col).isNotNull())
        .withColumn("_buckets",
            F.expr("sequence(date_trunc('hour', _t_start), date_trunc('hour', _t_end), interval 1 hour)"))
        .select("_flight_id", F.col(station_col).cast("string").alias("_st"),
                "_t_start", "_t_end", F.explode("_buckets").alias("_bucket_hour")))

    rolled = (bucketed.join(wsrc,
                (bucketed["_st"] == wsrc["STATION"]) &
                (bucketed["_bucket_hour"] == wsrc["_weather_hour"]),
                "inner")
        .filter(F.col("_obs_date").between(F.col("_t_start"), F.col("_t_end")))
        .groupBy("_flight_id")
        .agg(*[F.expr(f"percentile_approx(`{c}`, 0.5)").alias(f"{prefix}{c}_roll")
               for c in ROLL_BASE_COLS]))

    out = df.join(rolled, on="_flight_id", how="left").drop("_rid", "_t_start", "_t_end")
    out = out.persist()
    out.count()
    df.unpersist()
    return out


def coalesce_rolling_impute(df):
    for pre in ("", "dest_"):
        for c in ROLL_BASE_COLS:
            snap = f"{pre}{c}_2h"
            roll = f"{pre}{c}_roll"
            if snap in df.columns and roll in df.columns:
                df = df.withColumn(snap, F.coalesce(F.col(snap), F.col(roll)))
    return df


# ============================================================================
# Graph features (airport network centrality)
# ============================================================================
def get_flights_graph(df, origin_col, dest_col, edge_weight_col=None):
    if edge_weight_col is None:
        edge_weights_df = df.groupBy(origin_col, dest_col).count()
        edges = edge_weights_df.select(F.col(origin_col), F.col(dest_col),
                                       F.col("count").alias("weight"))
    else:
        edge_weights_df = df.groupBy(origin_col, dest_col).agg(F.sum(edge_weight_col).alias("weight"))
        edges = edge_weights_df.select(F.col(origin_col), F.col(dest_col), F.col("weight"))
    edge_list = edges.collect()
    nodes = set()
    for row in edge_list:
        nodes.add(row[origin_col]); nodes.add(row[dest_col])
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    for row in edge_list:
        G.add_edge(row[origin_col], row[dest_col], weight=row["weight"])
    return G


def get_graphy_features_per_airport(G):
    in_deg   = dict(G.in_degree())
    out_deg  = dict(G.out_degree())
    deg      = dict(G.degree())
    pr       = nx.pagerank(G, weight="weight")
    bet      = nx.betweenness_centrality(G, weight="weight")
    close    = nx.closeness_centrality(G)
    nodes = list(G.nodes())
    return pd.DataFrame({
        "airport":     nodes,
        "inDegree":    [in_deg.get(n, 0)  for n in nodes],
        "outDegree":   [out_deg.get(n, 0) for n in nodes],
        "degree":      [deg.get(n, 0)     for n in nodes],
        "pageRank":    [pr.get(n, 0)      for n in nodes],
        "betweenness": [bet.get(n, 0)     for n in nodes],
        "closeness":   [close.get(n, 0)   for n in nodes],
    })


def join_graphy_features_to_flights(df, airport_features):
    origin_features = spark.createDataFrame(airport_features).withColumnRenamed("airport", "ORIGIN")
    for col in airport_features.columns:
        if col != "airport":
            origin_features = origin_features.withColumnRenamed(col, f"ORIGIN_{col}")
    dest_features = spark.createDataFrame(airport_features).withColumnRenamed("airport", "DEST")
    for col in airport_features.columns:
        if col != "airport":
            dest_features = dest_features.withColumnRenamed(col, f"DEST_{col}")
    return (df.join(F.broadcast(origin_features), on="ORIGIN", how="left")
              .join(F.broadcast(dest_features),   on="DEST",   how="left"))


def add_graphy_features(df):
    G      = get_flights_graph(df, "ORIGIN", "DEST")
    feats  = get_graphy_features_per_airport(G)
    return join_graphy_features_to_flights(df, feats)

## 5. `build_features` orchestrator

In [0]:
def build_features(df_raw, df_weather_src, cal_df_broadcast):
    """Full FE pipeline: raw OTPW+weather -> feature-engineered DataFrame ready for encoding."""
    df = normalize_input_types(df_raw)
    df = remove_null_targets(df)                              
    df = add_departure_time_features(df)
    df = add_holiday_features(df, cal_df_broadcast)
    df = add_weather_cols(df, df_weather_src)
    df = clean_weather_num(weather_num_cols, df)
    df = add_rolling_weather_medians(df, df_weather_src, "STATION",         "")
    df = add_rolling_weather_medians(df, df_weather_src, "dest_station_id", "dest_")
    df = coalesce_rolling_impute(df)
    df = df.checkpoint()

    df = get_precip_features(df)
    df = get_sky_features(df)
    df = get_pressure_spread(df)
    df = add_leg_features(df)
    df = add_graphy_features(df)
    return df

## 6. Run FE (or load intermediate)

Write `features_raw_{size}.parquet` once. Set `REBUILD_RAW = True` at the top of the notebook to force a rebuild.

In [0]:
try:
    dbutils.fs.ls(RAW_PATH)
    raw_exists = True
except Exception:
    raw_exists = False

if raw_exists and not REBUILD_RAW:
    print(f"Loading existing raw features from {RAW_PATH}")
    df_features_raw = spark.read.parquet(RAW_PATH)
else:
    print(f"Building raw features from scratch...")
    df_features_raw = build_features(df_otpw_raw, df_weather, cal_df)
    print(f"Writing raw features to {RAW_PATH}")
    df_features_raw.write.mode("overwrite").parquet(RAW_PATH)
    df_features_raw = spark.read.parquet(RAW_PATH)  # reload for clean lineage

print(f"Raw features rows: {df_features_raw.count():,}")
print(f"Raw features cols: {len(df_features_raw.columns)}")

Building raw features from scratch...
Writing raw features to dbfs:/student-groups/Group_1_1/rolling/features_raw_12m.parquet
Raw features rows: 5,726,181
Raw features cols: 318


## 7. Column selection + train/test split

Keep only the columns needed for modeling. The unified checkpoint holds identity columns, all numeric features (shared across model families), and every string categorical (kept as-is here — encoding happens in section 8).

In [0]:
def select_model_columns(df):
    keep = list(dict.fromkeys(
        identity_cols
        + schedule_cols + flight_cols + time_hol_numeric + graphy_cols
        + weather_num_cols + weather_trend_numeric + weather_flag_cols + ["sky_ceiling_ft", "dest_sky_ceiling_ft", "spread_2h"]
        + STRING_CAT_COLS
        + ["DAY_OF_WEEK"]
    ))
    keep = [c for c in keep if c in df.columns]
    missing = set(
        schedule_cols + flight_cols + time_hol_numeric + graphy_cols
        + weather_num_cols + weather_trend_numeric + weather_flag_cols + ["sky_ceiling_ft", "dest_sky_ceiling_ft", "spread_2h"]
        + STRING_CAT_COLS + ["DAY_OF_WEEK", label_col]
    ) - set(df.columns)
    if missing:
        print(f"WARNING: missing expected columns: {missing}")
    return df.select(*keep)


df_model = select_model_columns(df_features_raw)
print(f"After column selection: {len(df_model.columns)} columns")

# Cast DAY_OF_WEEK to int
df_model = df_model.withColumn("DAY_OF_WEEK", F.col("DAY_OF_WEEK").cast("int"))

# Time-based train/test split
if DATASET_SIZE == "12m":
    # Split by QUARTER: train = Q1-Q3, test = Q4
    train_df = df_model.filter(F.col("QUARTER").cast("int").isin([1, 2, 3]))
    test_df  = df_model.filter(F.col("QUARTER").cast("int") == 4)
    print("Split by QUARTER for 12m: train = Q1-Q3, test = Q4")
elif DATASET_SIZE == "60m":
    # Split by YEAR: train = 2015-2018, test = 2019
    train_df = df_model.filter(F.col("FL_DATE") < "2019-01-01")
    test_df  = df_model.filter(F.col("FL_DATE") >= "2019-01-01")
    print("Split by YEAR for 60m: train = 2015-2018, test = 2019")
else:
    raise ValueError(f"Unsupported DATASET_SIZE: {DATASET_SIZE}")

print(f"Train rows: {train_df.count():,}   Test rows: {test_df.count():,}")

After column selection: 77 columns
Split by QUARTER for 12m: train = Q1-Q3, test = Q4
Train rows: 4,308,892   Test rows: 1,417,289


## 8. Fit encoders on train, transform both sides

All fit-then-transform steps happen here. **Fit uses `train_df` only** to avoid target leakage from test-set categorical distributions; transforms are applied to both. `handleInvalid="keep"` ensures rare unseen categories in test get a safe "unknown" index rather than crashing.

**What's NOT here** (belongs in each model's own notebook):
- **Imputer** (median) — must refit inside each CV fold.
- **StandardScaler / RobustScaler** — model-specific + must refit per fold.
- **VectorAssembler** — cheap to add downstream; makes feature ablation trivial.

In [0]:
# --- Bucketizers (linear-only; trees split on continuous natively) ----------
bucketizers = [
    Bucketizer(splits=splits, inputCol=inp, outputCol=out, handleInvalid="keep")
    for (inp, out, splits) in BUCKET_SPECS
]

# Bucketizer needs spread_2h_imp -> we median-impute spread_2h only for the
# bucket definition. This median is fit on TRAIN only.
#bucket_imputer = Imputer(
    #inputCols=["spread_2h"], outputCols=["spread_2h_imp"], strategy="median",
#)

# --- StringIndexers (feed BOTH tree family and linear-OHE stage) -----------
string_indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx",
                  handleInvalid="keep", stringOrderType="alphabetAsc")
    for c in STRING_CAT_COLS
]

# --- OneHotEncoders (linear-only) ------------------------------------------
# Categorical string columns get OHE'd from their `_idx` output.
cat_ohe = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in STRING_CAT_COLS],
    outputCols=[f"{c}_oh" for c in STRING_CAT_COLS],
    handleInvalid="keep",
)

# Bucketized weather features get their own OHE.
bucket_ohe = OneHotEncoder(
    inputCols=weather_bucket_bins,
    outputCols=weather_bucket_ohs,
    handleInvalid="keep",
)

# DAY_OF_WEEK: OHE the raw integer directly (1=Monday, 7=Sunday)
dow_ohe = OneHotEncoder(
    inputCols=["DAY_OF_WEEK"], outputCols=["dow_oh"], handleInvalid="keep",
)

# --- Assemble encoder pipeline ---------------------------------------------
encoder_pipeline = Pipeline(stages=[
    *bucketizers,
    *string_indexers,
    cat_ohe,
    bucket_ohe,
    dow_ohe,
])

print("Fitting encoders on train_df...")
encoder_model = encoder_pipeline.fit(train_df)
print("Done. Transforming train + test...")

train_encoded = encoder_model.transform(train_df)
test_encoded  = encoder_model.transform(test_df)

print(f"Encoded train cols: {len(train_encoded.columns)}")
print(f"Encoded test cols:  {len(test_encoded.columns)}")

Fitting encoders on train_df...
Done. Transforming train + test...
Encoded train cols: 100
Encoded test cols:  100


## 9. Save unified train/test checkpoints

One parquet each. Downstream model notebooks load these directly and select the columns their model family needs.

In [0]:
# Drop the internal `spread_2h_imp` column used only to feed the spread bucketizer;
# each model notebook will refit its own imputer inside its CV folds.
drop_internal = ["spread_2h_imp"]
train_out = train_encoded.drop(*[c for c in drop_internal if c in train_encoded.columns])
test_out  = test_encoded.drop( *[c for c in drop_internal if c in test_encoded.columns])

print(f"Writing train checkpoint to {TRAIN_PATH}")
train_out.write.mode("overwrite").parquet(TRAIN_PATH)

print(f"Writing test checkpoint to {TEST_PATH}")
test_out.write.mode("overwrite").parquet(TEST_PATH)

print("Done.")

Writing train checkpoint to dbfs:/student-groups/Group_1_1/rolling/features_12m_train.parquet
Writing test checkpoint to dbfs:/student-groups/Group_1_1/rolling/features_12m_test.parquet
Done.


## 10. Verification

Reload both checkpoints and confirm schemas + row counts. Checks if anything's wrong with the encoder outputs (e.g., a missing column, wrong type) before progressing to model building.

In [0]:
train_reload = spark.read.parquet(TRAIN_PATH)
test_reload  = spark.read.parquet(TEST_PATH)

print(f"Train rows: {train_reload.count():,}  cols: {len(train_reload.columns)}")
print(f"Test rows:  {test_reload.count():,}   cols: {len(test_reload.columns)}")

# --- Confirm all expected columns present ----------------------------------
expected_shared = (
    identity_cols + schedule_cols + flight_cols + time_hol_numeric + graphy_cols
    + weather_num_cols + weather_trend_numeric + weather_flag_cols
    + weather_lin_continuous + ["DAY_OF_WEEK"]
)
expected_tree_cats   = [f"{c}_idx" for c in STRING_CAT_COLS]
expected_linear_cats = ([f"{c}_oh" for c in STRING_CAT_COLS] + weather_bucket_ohs + ["dow_oh"])
expected = set(expected_shared + expected_tree_cats + expected_linear_cats + weather_bucket_bins)

missing_train = expected - set(train_reload.columns)
missing_test  = expected - set(test_reload.columns)
print(f"Missing from train ({len(missing_train)}): {sorted(missing_train)}")
print(f"Missing from test  ({len(missing_test)}): {sorted(missing_test)}")

# --- Column type sanity: OHE columns should be Vector, _idx should be double
from pyspark.ml.linalg import VectorUDT
for c in expected_linear_cats:
    if c in train_reload.columns:
        dtype = dict(train_reload.dtypes)[c]
        if "vector" not in dtype.lower() and "struct" not in dtype.lower():
            print(f"WARN: expected {c} to be a Vector, got {dtype}")

print("Verification complete.")

Train rows: 4,308,892  cols: 100
Test rows:  1,417,289   cols: 100
Missing from train (0): []
Missing from test  (0): []
Verification complete.
